In [ ]:
%load_ext watermark


In [ ]:
import os

from IPython.display import display
from matplotlib import pyplot as plt
import polars as pl
import seaborn as sns
from teeplot import teeplot as tp

import pylib  # noqa: F401


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-31-benchmark")
teeplot_subdir


## Prep Data


In [ ]:
df = pl.read_csv("http://osf.io/download/utv5w").to_pandas()
display(df.describe()), display(df.head()), display(df.tail());


## Example Plot


In [ ]:
df["throughput"] = df["n_leaves"] / df["seconds"]


In [ ]:
# 1. Extract the filtered data so we can easily reference it for labeling
plot_df = df[
    ~df["operation"].str.contains("bytes")
    & ~df["operation"].str.contains("save")
    & ~df["operation"].str.contains("load")
    & (df["library"] != "phyloframe")
    & (df["library"] != "phyloframe (in-memory)")
].copy()
plot_df["tips"] = plot_df["n_leaves"]
plot_df["tips per sec"] = plot_df["throughput"]

unique_libs = plot_df["library"].unique()
palette = sns.color_palette("Dark2", n_colors=len(unique_libs))
with tp.teed(
    sns.relplot,
    data=plot_df,
    x="tips",
    y="tips per sec",
    hue="library",
    col="operation",
    col_order=[
        "inorder",
        "levelorder",
        "postorder",
        "preorder",
        "topological_order",
        "make_newick",
        "parse_newick",
        "mrca_allpairs",
        "pairwise_dist",
    ],
    col_wrap=3,
    facet_kws={
        "sharex": False,
        "sharey": False,
    },
    kind="line",
    legend=True,
    palette=palette,
    teeplot_subdir=teeplot_subdir,
) as teed:

    # 2. Map colors to libraries so the text labels match the line colors
    color_dict = dict(zip(unique_libs, palette))

    # 3. Define a labeling function to map over each facet
    def label_line_ends(data, **kwargs):
        ax = plt.gca()
        maxmax = 0
        for library, group_df in (
            data.dropna(subset="tips per sec").groupby("library")
        ):
            if group_df.empty:
                continue

            # Find the coordinates for the largest x value (n_leaves)
            max_x_idx = group_df["tips"].idxmax()
            max_x = group_df.loc[max_x_idx, "tips"]
            max_y = group_df.loc[max_x_idx, "tips per sec"]
            maxmax = max(maxmax, max_y)

        for library, group_df in (
            data.dropna(subset="tips per sec").groupby("library")
        ):
            if group_df.empty:
                continue

            # Find the coordinates for the largest x value (n_leaves)
            max_x_idx = group_df["tips"].idxmax()
            max_x = group_df.loc[max_x_idx, "tips"]
            max_y = group_df.loc[max_x_idx, "tips per sec"]
            if max_y < maxmax / 2.5:
                continue

            if library == "phyloframe (streaming+i32)" and max_y == maxmax:
                suffix = "\n"
            else:
                suffix = ""
            # Place the text at the end of the line
            ax.annotate(
                (
                    library
                    .replace("streaming+i32+", "")
                    .replace("streaming+i32", "")
                    .replace("()", "")
                    .replace(
                        "phyloframe (csr)", "\nphyloframe (csr)"
                    )
                    .replace(
                        "phyloframe (child/sib)", "phyloframe (sib)\n"
                    )
                    .replace("phyloframe", "pf")
                    .replace("compacttree", "ctree")
                    + suffix
                ),
                xy=(max_x, max_y),
                xytext=(5, 0), # Offset 5 points to the right
                textcoords="offset points",
                ha="left",
                va="center",
                color=color_dict.get(library, "black"),
                fontweight="bold",
                fontsize=9,
                clip_on=False,
            )

    # 4. Apply the function to the underlying FacetGrid
    teed.map_dataframe(label_line_ends)

    teed.fig.set_size_inches(8, 6)
    teed.fig.tight_layout()
    teed.set_titles(col_template="{col_name}")
    teed.set(xscale="log")

    sns.move_legend(
        teed, "lower center",
        bbox_to_anchor=(.5, 1), ncol=3, title=None, frameon=False,
    )


In [ ]:
# 1. Extract the filtered data so we can easily reference it for labeling
plot_df = df[
    df["operation"].str.contains("bytes")
    & (df["library"] != "phyloframe")
    & (df["library"] != "phyloframe (in-memory)")
    & (df["n_leaves"] > 100_000)
].copy()
plot_df["tips"] = plot_df["n_leaves"]
plot_df["tips per byte"] = df["throughput"]
print(plot_df.describe())

unique_libs = plot_df["library"].unique()
palette = sns.color_palette("Dark2", n_colors=len(unique_libs))
with tp.teed(
    sns.relplot,
    data=plot_df,
    x="tips",
    y="tips per byte",
    hue="library",
    col="operation",
    col_order=[
        "memory_bytes",
    ],
    col_wrap=3,
    facet_kws={
        "sharex": False,
        "sharey": False,
    },
    kind="line",
    legend=False,
    palette=palette,
    teeplot_subdir=teeplot_subdir,
) as teed:

    # 2. Map colors to libraries so the text labels match the line colors
    color_dict = dict(zip(unique_libs, palette))

    # 3. Define a labeling function to map over each facet
    def label_line_ends(data, **kwargs):
        ax = plt.gca()
        maxmax = 0
        for library, group_df in (
            data.dropna(subset="tips per byte").groupby("library")
        ):
            if group_df.empty:
                continue

            # Find the coordinates for the largest x value (n_leaves)
            max_x_idx = group_df["tips"].idxmax()
            max_x = group_df.loc[max_x_idx, "tips"]
            max_y = group_df.loc[max_x_idx, "tips per byte"]
            maxmax = max(maxmax, max_y)

        for library, group_df in (
            data.dropna(subset="tips per byte").groupby("library")
        ):
            if group_df.empty:
                continue

            # Find the coordinates for the largest x value (n_leaves)
            max_x_idx = group_df["tips"].idxmax()
            max_x = group_df.loc[max_x_idx, "tips"]
            max_y = group_df.loc[max_x_idx, "tips per byte"]
            # if max_y < maxmax / 2.5:
            #     continue

            # Place the text at the end of the line
            ax.annotate(
                (
                    library
                    .replace("streaming+i32+", "")
                    .replace("streaming+i32", "")
                    .replace("()", "")
                    .replace(
                        "phyloframe (csr)", "\nphyloframe (csr)"
                    )
                    .replace(
                        "phyloframe (child/sib)", "phyloframe (sib)\n"
                    )
                    .replace("phyloframe", "pf")
                    .replace("compacttree", "ctree")
                    .replace("biopython", "\nbiopython")
                    .replace("treeswift", "treeswift\n")
                ),
                xy=(max_x, max_y),
                xytext=(5, 0), # Offset 5 points to the right
                textcoords="offset points",
                ha="left",
                va="center",
                color=color_dict.get(library, "black"),
                fontweight="bold",
                fontsize=9,
                clip_on=False,
            )

    # 4. Apply the function to the underlying FacetGrid
    teed.map_dataframe(label_line_ends)

    teed.fig.set_size_inches(5, 4)
    teed.fig.tight_layout()
    teed.set_titles(col_template="")
    teed.set(xscale="log")
    teed.set(ylim=(0, None))
